# cv-data-lesson PyPI総合受け入れテスト

授業用Notebookとは別の確認用Notebookです。対象版 `cv-data-lesson==0.1.5` をPyPI公開後の新しいGoogle Colabランタイムへ導入し、仕様書にあるデータ取得、ラベル、split、bbox、ResNet、DETR、SAM、画像生成のCPU/GPU自動切替と入力境界を一括確認します。

CPUランタイムとT4 GPUランタイムで、このNotebookをそれぞれ最初から実行してください。初回は固定ReleaseのデータとHugging Faceの固定revisionモデルを取得するため、長時間・大容量になるセルがあります。モデル重みとデータはColabのキャッシュにだけ保存され、リポジトリやPyPIへ追加されません。

OCRはPaddlePaddleとPyTorch/Diffusersの環境を分ける仕様のため、最後のOCR専用セクションを新しいランタイムで実行します。既存の [OCR受け入れテスト](ocr_acceptance_test.ipynb) と同じ確認です。

In [ ]:
import sys
print("Python:", sys.version)
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Python 3.11〜3.13のランタイムを選択してください"

## 公開PyPI版を導入

最初のセルは、利用者と同じ通常インストールです。次のセルでモデル確認用の追加依存を、同じ `0.1.5` に固定して導入します。Colab既存パッケージとの依存関係警告が表示されることがありますが、最後にバージョンと実行結果を確認します。

In [ ]:
%pip install -q --upgrade "cv-data-lesson==0.1.5"

In [ ]:
%pip install -q --upgrade "cv-data-lesson[dataframe,display,torch,vision,diffusion]==0.1.5"

In [ ]:
import gc
import importlib.metadata as metadata
import tempfile
import warnings
from pathlib import Path

import torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

import cvlesson
from cvlesson import datasets, generation, models, vision

LIBRARY_VERSION = metadata.version("cv-data-lesson")
HAS_GPU = bool(torch.cuda.is_available())
MODEL_CACHE = Path("/content/cvlesson-pypi-model-cache")
DATA_CACHE = Path("/content/cvlesson-pypi-data-cache")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
DATA_CACHE.mkdir(parents=True, exist_ok=True)
print("cv-data-lesson:", LIBRARY_VERSION)
print("module:", cvlesson.__file__)
print("runtime:", "T4/GPU" if HAS_GPU else "CPU")
assert LIBRARY_VERSION == "0.1.5"
assert cvlesson.__file__ is not None
print("ResNet devices:", models.available_devices())
print("Vision devices:", vision.available_devices())
print("Generation devices:", generation.available_devices())
assert models.available_devices()["cpu"] is True
assert vision.available_devices()["cpu"] is True
assert generation.available_devices()["cpu"] is True
if HAS_GPU:
    assert models.available_devices()["cuda"] is True
    assert vision.available_devices()["cuda"] is True
    assert generation.available_devices()["cuda"] is True
print("公開版・実行環境・利用可能device: OK")

## カタログと日本語ラベル

ラベル表はデータ本体をダウンロードせずに確認します。元ラベルは機械的な対応や出典確認に使うため残し、日本語ラベルは授業で意味が伝わる表現にします。

In [ ]:
catalog = datasets.list_available()
catalog_ids = {item["id"] for item in catalog}
assert len(catalog) >= 24
for required_id in ("geoset_ennea", "rdd2020_japan", "fruitseg30", "indoor_waste_image", "fresh_rotten_fruits"):
    assert required_id in catalog_ids, required_id

label_expectations = {
    "orange_fruit": {
        "cankered_orange": "傷んだオレンジ",
        "fresh_orange": "新鮮なオレンジ",
        "rotten_orange": "腐ったオレンジ",
    },
    "fruitseg30": {
        "Apple_Gala": "りんご（ガラ種）",
        "Apple_Golden Delicious": "りんご（ゴールデンデリシャス種）",
        "Green Coconut": "緑色のココナッツ",
        "Malta": "マルタオレンジ",
    },
    "indoor_waste_image": {"shell": "ナッツの殻"},
    "fire_recognition_image": {
        "Artificial_Fire": "擬似的な炎",
        "Fire": "炎",
        "Safe_Fire": "日常で扱われる火",
    },
    "fresh_rotten_fruits": {
        "fresh_apple": "新鮮なリンゴ",
        "rotten_apple": "傷んだリンゴ",
    },
    "agrifreshnet": {
        "Fresh Orange(1-9)": "新鮮なオレンジ",
        "Rotten Orange(20-35)": "傷んだオレンジ",
        "Semi fresh Orange(9-20)": "少し傷んだオレンジ",
    },
}
for dataset_id, expected in label_expectations.items():
    source = datasets._source(dataset_id)
    labels_ja = dict(zip(source.classes, source.class_names_ja))
    labels_ja.update(source.label_names_ja or {})
    for raw_label, expected_ja in expected.items():
        assert raw_label in source.classes, (dataset_id, raw_label)
        assert labels_ja[raw_label] == expected_ja, (dataset_id, raw_label, labels_ja[raw_label])
print("catalog entries:", len(catalog), "selected raw/Japanese labels: OK")

## GeoSetEnnea・RDD2020の固定Release取得と形式確認

このセルは約650MBの固定artifactを取得します。`splits`と`to_df()["split"]`は `train`/`val`/`test`へ統一し、`to_df()["source_split"]`に配布元の表記を残します。

In [ ]:
geo = datasets.get("geoset_ennea", cache_dir=DATA_CACHE)
rdd = datasets.get("rdd2020_japan", cache_dir=DATA_CACHE)
assert geo.splits == {"train": 84, "val": 22, "test": 30}
assert rdd.splits == {"train": 6299, "val": 802, "test": 799}
assert geo.root.is_dir() and rdd.root.is_dir()
print("GeoSetEnnea:", geo.splits)
print("RDD2020 Japan:", rdd.splits)
print("fixed Release download and normalized split counts: OK")

In [ ]:
geo_df = geo.to_df()
rdd_df = rdd.to_df()
geo_splits = set(geo_df["split"].unique())
rdd_splits = set(rdd_df["split"].unique())
geo_source_splits = set(geo_df["source_split"].unique())
rdd_source_splits = set(rdd_df["source_split"].unique())
print("GeoSetEnnea normalized/source splits:", geo_splits, geo_source_splits)
print("RDD2020 normalized/source splits:", rdd_splits, rdd_source_splits)
assert geo_splits == {"train", "val", "test"}
assert rdd_splits == {"train", "val", "test"}
assert geo_source_splits == {"train", "val", "test"}
assert rdd_source_splits == {"train", "valid", "test"}
assert {"image_path", "label_file", "split", "boxes"} <= set(rdd_df.columns)
all_boxes = [box for boxes in rdd_df["boxes"] for box in boxes]
assert len(rdd_df) == 7900
assert all(box.get("class_name") and box.get("class_name_ja") for box in all_boxes)
bbox_count = 0
for boxes in rdd_df["boxes"]:
    for box in boxes:
        bbox_count += 1
        values = [float(box[key]) for key in ("x_center", "y_center", "width", "height")]
        assert all(value == value and abs(value) != float("inf") for value in values)
        x_center, y_center, width, height = values
        assert width > 0 and height > 0
        assert -1e-9 <= x_center - width / 2 <= 1 + 1e-9
        assert -1e-9 <= y_center - height / 2 <= 1 + 1e-9
        assert -1e-9 <= x_center + width / 2 <= 1 + 1e-9
        assert -1e-9 <= y_center + height / 2 <= 1 + 1e-9
assert bbox_count == 16469
cached = datasets.get("rdd2020_japan", cache_dir=DATA_CACHE, allow_download=False)
assert cached.root == rdd.root
print("DataFrame schema, original split notation, Japanese labels, cache reuse: OK")
print("RDD2020 bbox count/range: OK (", bbox_count, ")")

# 実データを実際に表示し、RDD2020の正解bboxを画像へ重ねて保存する。
from cvlesson import display as cv_display
from cvlesson.vision import Detection

geo_figure = geo.show_examples(n=4, split="train")
assert geo_figure is not None
print("GeoSetEnnea real-image display: OK")

rdd_row = rdd_df[rdd_df["boxes"].map(bool)].iloc[0]
with Image.open(rdd_row["image_path"]) as opened:
    actual_rdd_image = opened.convert("RGB")
actual_detections = []
image_width, image_height = actual_rdd_image.size
for box in rdd_row["boxes"]:
    x_center, y_center = float(box["x_center"]), float(box["y_center"])
    box_width, box_height = float(box["width"]), float(box["height"])
    actual_detections.append(Detection(
        label=f"正解 {box['class_name_ja']}",
        score=1.0,
        box=((x_center - box_width / 2) * image_width,
             (y_center - box_height / 2) * image_height,
             (x_center + box_width / 2) * image_width,
             (y_center + box_height / 2) * image_height),
    ))
display(actual_rdd_image)
actual_overlay = cv_display.show(actual_rdd_image, actual_detections)
actual_overlay_path = cv_display.save(
    actual_rdd_image,
    Path("/content/cvlesson-pypi-real-output/rdd_annotation_overlay.png"),
    actual_detections,
)
assert actual_overlay.size == actual_rdd_image.size
assert actual_overlay_path.is_file()
print("RDD2020 real-image display/annotation overlay/save: OK")

## ResNet分類・回帰（`device="auto"`）

小さな合成画像で実際に `fit`、進捗表示、`predict`、`score`を確認します。事前学習重みは使わず、固定特徴抽出器と出力層の学習経路だけを短時間で通します。

In [ ]:
classification_images = [Image.new("RGB", (96, 96), color) for color in ("red", "red", "blue", "blue")]
classification_labels = ["赤", "赤", "青", "青"]
classifier = models.ResNetClassifier(device="auto", epochs=1, batch_size=2, weights=None, verbose=True)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always")
    classifier.fit(classification_images, classification_labels)
assert classifier.device == ("cuda" if HAS_GPU else "cpu")
if not HAS_GPU:
    assert any("CPU" in str(item.message) for item in captured)
predictions = classifier.predict(classification_images)
probabilities = classifier.predict_proba(classification_images)
assert len(predictions) == 4
assert probabilities.shape == (4, 2)
assert 0.0 <= classifier.score(classification_images, classification_labels) <= 1.0
print("ResNet classification device:", classifier.device)
print("ResNet classification: OK")

# 実データの画像パスと原ラベルを、そのまま分類APIへ渡す。
real_labels = list(geo_df["label"].drop_duplicates().head(2))
real_rows = geo_df[geo_df["label"].isin(real_labels)].head(4)
assert len(real_rows) >= 2 and len(set(real_rows["label"])) == 2
real_classifier = models.ResNetClassifier(device="auto", epochs=1, batch_size=1, weights=None, verbose=True)
real_classifier.fit(real_rows["image_path"].tolist(), real_rows["label"].tolist())
real_predictions = real_classifier.predict(real_rows["image_path"].tolist())
assert len(real_predictions) == len(real_rows)
print("ResNet real-data path/label prediction: OK")

In [ ]:
regression_images = [
    (Image.new("RGB", (96, 96), "black"), Image.new("RGB", (96, 96), "white")),
    (Image.new("RGB", (96, 96), "white"), Image.new("RGB", (96, 96), "black")),
]
regressor = models.ResNetRegressor(device="auto", epochs=1, batch_size=1, weights=None, verbose=True)
regressor.fit(regression_images, [0.25, 0.75])
regression_predictions = regressor.predict(regression_images)
assert regressor.device == ("cuda" if HAS_GPU else "cpu")
assert regression_predictions.shape == (2,)
assert isinstance(regressor.score(regression_images, [0.25, 0.75]), float)
print("ResNet regression device:", regressor.device)
print("ResNet regression: OK")
del classifier, regressor
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()

## DETR物体検出・SAM領域分割（`device="auto"`）

固定revisionのDETRとSAMを実際に読み込みます。CPUではT4よりかなり時間がかかります。DETRはCOCOカテゴリの閉集合検出、SAMは点または枠からマスク候補を作るモデルで、任意の日本語物体名を理解するテストではありません。

In [ ]:
vision_image = Image.new("RGB", (256, 256), "white")
vision_draw = ImageDraw.Draw(vision_image)
vision_draw.rectangle((48, 48, 208, 208), fill="red")
detector = vision.load_detector(device="auto", cache_dir=MODEL_CACHE)
assert detector.device == ("cuda" if HAS_GPU else "cpu")
detections = detector.predict(vision_image, threshold=0.0)
assert isinstance(detections, list)
assert all(0.0 <= result.score <= 1.0 for result in detections)
synthetic_detection_view = cv_display.show(vision_image, detections)
synthetic_detection_path = cv_display.save(
    vision_image, Path("/content/cvlesson-pypi-real-output/detr_synthetic_overlay.png"), detections
)
assert synthetic_detection_view.size == vision_image.size
assert synthetic_detection_path.is_file()
print("DETR device:", detector.device, "detections:", len(detections))
print("DETR prediction/display/save: OK")

In [ ]:
segmenter = vision.load_segmenter(device="auto", cache_dir=MODEL_CACHE)
assert segmenter.device == ("cuda" if HAS_GPU else "cpu")
segmentation = segmenter.predict(vision_image, box=(48, 48, 208, 208))
assert segmentation.mask.shape == (256, 256)
assert segmentation.mask.dtype == bool
assert segmentation.prompt_type == "box"
print("SAM device:", segmenter.device, "mask score:", segmentation.score)
print("SAM prediction: OK")

# 固定Releaseから取得した実道路画像を、DETR/SAMと共通表示APIへ渡す。
real_detections = detector.predict(actual_rdd_image, threshold=0.5)
assert isinstance(real_detections, list)
real_detection_view = cv_display.show(actual_rdd_image, real_detections)
real_detection_path = cv_display.save(
    actual_rdd_image, Path("/content/cvlesson-pypi-real-output/detr_real_overlay.png"), real_detections
)
assert real_detection_view.size == actual_rdd_image.size
assert real_detection_path.is_file()
actual_segmentation = segmenter.predict(actual_rdd_image, box=actual_detections[0].box)
assert actual_segmentation.mask.shape == (actual_rdd_image.height, actual_rdd_image.width)
sam_view = cv_display.show(actual_rdd_image, actual_segmentation)
sam_path = cv_display.save(
    actual_rdd_image, Path("/content/cvlesson-pypi-real-output/sam_real_overlay.png"), actual_segmentation
)
combined_view = cv_display.show(actual_rdd_image, [*real_detections, actual_segmentation])
combined_path = cv_display.save(
    actual_rdd_image, Path("/content/cvlesson-pypi-real-output/detr_sam_real_overlay.png"),
    [*real_detections, actual_segmentation],
)
assert sam_view.size == actual_rdd_image.size
assert combined_view.size == actual_rdd_image.size
assert sam_path.is_file() and combined_path.is_file()
print("DETR/SAM real-image visualization and save: OK; detections:", len(real_detections))
del detector, segmenter, real_classifier
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()

## 画像生成（`model="auto"`, `device="auto"`）

GPUではSDXL、CPUでは固定revisionのTiny-SDを選びます。LoRAなしで共通のprompt、seed、steps、guidance、画像サイズを通し、CPU用Tiny-SDの非対応引数は次のセルで拒否されることを確認します。

In [ ]:
with warnings.catch_warnings(record=True) as generation_warnings:
    warnings.simplefilter("always")
    generator = generation.load(model="auto", device="auto", lora=None, cache_dir=MODEL_CACHE)
assert generator.device == ("cuda" if HAS_GPU else "cpu")
assert generator.model == ("sdxl" if HAS_GPU else "tiny-sd")
if not HAS_GPU:
    assert any("Tiny-SD" in str(item.message) for item in generation_warnings)
generated = generator.generate(
    "a simple red apple on a white table",
    seed=42,
    num_inference_steps=2,
    guidance_scale=5.0,
    width=256,
    height=256,
)
assert isinstance(generated, Image.Image)
assert generated.size == (256, 256)
display(generated)
print("generation model/device:", generator.model, generator.device)
print("text-to-image and common parameters: OK")

In [ ]:
if not HAS_GPU:
    try:
        generation.load(model="tiny-sd", device="cpu", lora="fluent_emoji_3d")
    except ValueError as error:
        print("Tiny-SD LoRA rejection: OK -", error)
    else:
        raise AssertionError("Tiny-SDがLoRA指定を受け入れました")
    try:
        generation.load(model="tiny-sd", device="cpu", task="image-to-image", local_files_only=True)
    except ValueError as error:
        print("Tiny-SD image-to-image rejection: OK -", error)
    else:
        raise AssertionError("Tiny-SDがimage-to-image指定を受け入れました")
else:
    print("T4ではTiny-SD境界はモデル取得なしのCPU条件テストを省略。単体テストで確認済み")
try:
    generation.load(model="sdxl", device="cpu")
except generation.GenerationError as error:
    print("explicit SDXL on CPU rejection: OK -", error)
else:
    raise AssertionError("SDXLのCPU明示指定が拒否されませんでした")

In [ ]:
if not HAS_GPU:
    try:
        models.ResNetClassifier(device="cuda", epochs=1, weights=None).fit(
            [Image.new("RGB", (64, 64), "red"), Image.new("RGB", (64, 64), "blue")],
            ["赤", "青"],
        )
    except models.ModelError as error:
        print("explicit ResNet GPU rejection: OK -", error)
    else:
        raise AssertionError("利用できないGPU指定がCPUへ黙って切り替わりました")
    try:
        vision.load_detector(device="cuda", local_files_only=True)
    except vision.VisionError as error:
        print("explicit DETR GPU rejection: OK -", error)
    else:
        raise AssertionError("利用できないDETR GPU指定が受け入れられました")
else:
    print("T4ではcuda/cuda:0を利用可能と環境セルで確認済み")

## OCR（専用ランタイムで実行）

ここから下は、PyTorch/Diffusersを入れたランタイムをいったん破棄し、新しいCPUまたはGPUランタイムで実行してください。PaddlePaddleはOS・CUDA別の公式パッケージを使う必要があり、同じランタイムでの併用は仕様上保証していません。

実行手順は [OCR受け入れテスト](ocr_acceptance_test.ipynb) にまとまっています。次のインストールセルは、PyPI公開版 `cv-data-lesson[ocr,dataframe,display]==0.1.5` をすでに導入するセルです。セルを書き換えずに実行し、CPU版PaddlePaddleを公式手順で導入します。OCRでは固定PP-OCRv5 mobileの読み込み、PNG/JPEG/WebP/Pillow Image、壊れた画像の `OCRError`、固定Releaseの日本語レシート画像を確認してください。

In [ ]:
# ここから下はランタイムを再起動してから実行する。CPU版PaddlePaddleの例。
%pip install -q "paddlepaddle==3.2.0" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
%pip install -q --upgrade "cv-data-lesson[ocr,dataframe,display]==0.1.5"

In [ ]:
from pathlib import Path
from importlib.metadata import version
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
from cvlesson import datasets, ocr, display as cv_display

print("cv-data-lesson:", version("cv-data-lesson"))
print("PaddlePaddle:", version("paddlepaddle"))
print("OCR devices:", ocr.available_devices())
assert version("cv-data-lesson") == "0.1.5"
assert ocr.available_devices()["cpu"] is True
OCR_CACHE = Path("/content/cvlesson-pypi-ocr-model-cache")
OCR_DATA_CACHE = Path("/content/cvlesson-pypi-ocr-data-cache")
reader = ocr.load(cache_dir=OCR_CACHE)
assert reader.device == "cpu"
print("fixed PP-OCRv5 mobile model load: OK")

In [ ]:
from cvlesson.ocr import OCR

class BoundaryPipeline:
    def __init__(self, scores):
        self.scores = scores

    def predict(self, *, input):
        return [{"res": {
            "rec_texts": ["境界テスト"],
            "rec_scores": self.scores,
            "rec_polys": [[[1, 1], [80, 1], [80, 20], [1, 20]]],
        }}]

boundary_image = Image.new("RGB", (96, 32), "white")
def assert_confidence_rejected(scores):
    fake_reader = OCR(BoundaryPipeline(scores), device="cpu", language="japan")
    try:
        fake_reader.predict(boundary_image)
    except ocr.OCRError:
        return
    raise AssertionError(f"不正な信頼度を受け入れました: {scores}")

for invalid_scores in ([], [-0.5], [1.2], [float("nan")]):
    assert_confidence_rejected(invalid_scores)
valid_reader = OCR(BoundaryPipeline([0.73]), device="cpu", language="japan")
valid_result = valid_reader.predict(boundary_image)
assert valid_result.lines[0].confidence == 0.73
print("OCR confidence boundary validation: OK")

In [ ]:
ocr_image = Image.new("RGB", (1200, 260), "white")
ocr_draw = ImageDraw.Draw(ocr_image)
font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = ImageFont.truetype(font_path, 92) if Path(font_path).exists() else ImageFont.load_default()
ocr_draw.text((40, 65), "TOTAL 1280 JPY", fill="black", font=font)
ocr_input_dir = Path("/content/cvlesson-pypi-ocr-inputs")
ocr_input_dir.mkdir(parents=True, exist_ok=True)
png_path = ocr_input_dir / "sample.png"
jpeg_path = ocr_input_dir / "sample.jpg"
webp_path = ocr_input_dir / "sample.webp"
ocr_image.save(png_path, format="PNG")
ocr_image.save(jpeg_path, format="JPEG")
ocr_image.save(webp_path, format="WEBP")
results = [reader.predict(path) for path in (png_path, jpeg_path, webp_path)]
pillow_result = reader.predict(ocr_image)
for result in results + [pillow_result]:
    assert isinstance(result.text, str)
    for line in result.lines:
        assert line.text.strip()
        assert 0.0 <= line.confidence <= 1.0
assert results[2].text.strip(), "WebPが空結果になっています"
display(ocr_image)
print("PNG/JPEG/WebP/Pillow Image OCR: OK")
print("WebP lines:", len(results[2].lines))

In [ ]:
broken_path = ocr_input_dir / "broken.webp"
broken_path.write_bytes(b"not an image")
try:
    reader.predict(broken_path)
except ocr.OCRError as error:
    print("broken image rejection: OK -", error)
else:
    raise AssertionError("壊れた画像が空のOCR結果として処理されました")

receipt_data = datasets.get("japanese_receipts_detailed", cache_dir=OCR_DATA_CACHE)
receipt_frame = receipt_data.to_df()
receipt_path = Path(receipt_frame.iloc[0]["image_path"])
with Image.open(receipt_path) as opened:
    receipt_image = opened.convert("RGB")
receipt_result_from_path = reader.predict(receipt_path)
receipt_result = reader.predict(receipt_image)
assert isinstance(receipt_result.text, str)
assert len(receipt_result.lines) == len(receipt_result_from_path.lines)
display(receipt_image)
receipt_overlay = cv_display.show(receipt_image, receipt_result)
assert receipt_overlay.size == receipt_image.size
print("fixed Japanese receipt path/Pillow OCR and overlay: OK")
print("receipt images:", len(receipt_frame), " / lines:", len(receipt_result.lines))

In [ ]:
# 共通表示APIの実動作を確認する。
import tempfile
import numpy as np
from PIL import Image
from cvlesson import display as cv_display
from cvlesson.ocr import OCRLine, OCRResult
from cvlesson.vision import Detection, Segmentation
display_fixture = Image.new("RGB", (128, 128), "white")
display_detection = Detection("fixture", 0.9, (16, 20, 96, 110))
display_mask = np.zeros((128, 128), dtype=bool)
display_mask[32:96, 32:96] = True
display_segmentation = Segmentation(display_mask, 0.8, "box", display_detection.box)
display_ocr = OCRResult((OCRLine("確認", 0.95, ((8, 8), (70, 8), (70, 28), (8, 28))),), "fixture")
rendered = cv_display.render(display_fixture, [display_detection, display_segmentation, display_ocr])
assert rendered.size == display_fixture.size
with tempfile.TemporaryDirectory() as temporary_directory:
    saved_path = cv_display.save(display_fixture, Path(temporary_directory) / "overlay.png", [display_detection, display_segmentation, display_ocr])
    assert saved_path.is_file()
shown = cv_display.show(display_fixture, display_detection)
assert shown.size == display_fixture.size
print("display render/show/save: OK")

## 判定

このNotebookの全セルと、専用ランタイムのOCRセクションが成功すれば、PyPI `0.1.5` の固定データ・ラベル・split・bbox、ResNet分類/回帰、DETR、SAM、画像生成のCPU/T4自動切替と境界、OCRの入力検証をColabで実測できます。これは実行した環境・モデル・入力に対する受け入れテストであり、全画像・全プロンプトでの精度や絶対的な無欠陥を保証するものではありません。失敗したセルの全文と、CPU/T4の別を記録して修理対象を切り分けます。